# **Aula 4 - CNN, RNN, LSTM, GAN e Autoencoders**

Na Aula 3, treinamos uma rede feedforward (FFNN) com Keras para o mesmo problema de spam das aulas anteriores: camadas densas, uma após a outra, sem nenhuma estrutura especial nos dados de entrada.

Isso funciona bem quando as entradas são um vetor de características (como as 3 do nosso e-mail). Mas imagens, sequências temporais e texto têm uma estrutura própria que uma rede densa comum ignora -- e explorar essa estrutura é exatamente o que motiva cada uma das arquiteturas desta aula. Vamos ver cada uma como resposta a uma limitação concreta da anterior.

## 1. Redes Neurais Feedforward (FFNN): o que já sabemos

A rede que construímos na Aula 3 (Keras, `Dense` + `Dense`) já é uma **FFNN**: a informação flui em uma única direção, da entrada para a saída, sem ciclos e sem memória de exemplos anteriores. É a arquitetura mais simples e serve como nossa referência de comparação para todas as outras desta aula.

**Limitação da FFNN:** se a entrada for uma imagem, por exemplo, uma camada `Dense` trataria cada pixel como uma característica independente, sem nenhuma noção de que pixels vizinhos estão relacionados espacialmente. Isso é o que a CNN resolve.

## 2. Redes Neurais Convolucionais (CNN)

### O que é uma CNN?

As CNNs são redes neurais projetadas para processar dados com **estrutura de grade**, como imagens. Em vez de conectar cada pixel a cada neurônio (como uma `Dense` faria), elas usam **filtros** que percorrem a imagem e aprendem padrões locais (bordas, texturas, formas) -- e reaproveitam o mesmo filtro em toda a imagem, o que reduz drasticamente o número de parâmetros comparado a uma rede densa.

### Estrutura de uma CNN

1. **Camada Convolucional**: aplica filtros (kernels) que deslizam sobre a imagem, extraindo características locais.
2. **Camada de Pooling**: reduz a dimensionalidade, preservando as características mais importantes (ex.: Max Pooling pega o valor máximo de cada região).
3. **Camada Densa (Fully Connected)**: nas etapas finais, combina as características extraídas para a classificação.

### Exemplo prático

Vamos classificar pequenas imagens 8x8 em duas classes: **linha vertical brilhante** ou **linha horizontal brilhante**, sobre um fundo com ruído. Diferente do exemplo original com uma única imagem, aqui vamos gerar 400 imagens (200 de cada classe) e separar 20% para teste, para termos uma acurácia real.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input
from sklearn.model_selection import train_test_split

seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)

n_por_classe = 200
tam = 8

def gera_linha_vertical(n):
    imgs = np.random.uniform(0, 0.3, (n, tam, tam))
    for i in range(n):
        col = np.random.randint(0, tam)
        imgs[i, :, col] = np.random.uniform(0.7, 1.0, tam)
    return imgs

def gera_linha_horizontal(n):
    imgs = np.random.uniform(0, 0.3, (n, tam, tam))
    for i in range(n):
        lin = np.random.randint(0, tam)
        imgs[i, lin, :] = np.random.uniform(0.7, 1.0, tam)
    return imgs

verticais = gera_linha_vertical(n_por_classe)
horizontais = gera_linha_horizontal(n_por_classe)

X = np.vstack([verticais, horizontais]).reshape(-1, tam, tam, 1)
y = np.concatenate([np.ones(n_por_classe), np.zeros(n_por_classe)])

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

modelo_cnn = Sequential()
modelo_cnn.add(Input(shape=(tam, tam, 1)))
modelo_cnn.add(Conv2D(8, kernel_size=(3, 3), activation='relu'))
modelo_cnn.add(MaxPooling2D(pool_size=(2, 2)))
modelo_cnn.add(Flatten())
modelo_cnn.add(Dense(8, activation='relu'))
modelo_cnn.add(Dense(1, activation='sigmoid'))
modelo_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Treinando a CNN...")
modelo_cnn.fit(X_treino, y_treino, epochs=30, verbose=0)
perda, acuracia = modelo_cnn.evaluate(X_teste, y_teste, verbose=0)
print(f"Acuracia no conjunto de teste: {acuracia:.2%}")

Treinando a CNN...


Acuracia no conjunto de teste: 100.00%


### Explicação do Código

- `Conv2D(8, kernel_size=(3,3))`: 8 filtros diferentes, cada um de 3x3 pixels, deslizando pela imagem para detectar padrões locais (como um trecho de linha).
- `MaxPooling2D(pool_size=(2,2))`: reduz a imagem pela metade em cada dimensão, mantendo os valores mais fortes.
- `Flatten` + `Dense`: depois de extrair as características espaciais, achatamos tudo em um vetor e aplicamos as mesmas camadas densas da Aula 3 para a classificação final.

**Aplicação real:** veja [`aplicacoes-de-negocio/05-visao-computacional-varejo.ipynb`](aplicacoes-de-negocio/05-visao-computacional-varejo.ipynb) e [`06-diagnostico-imagem-medica.ipynb`](aplicacoes-de-negocio/06-diagnostico-imagem-medica.ipynb) para exemplos completos de CNN em problemas de negócio.

**Limitação da CNN:** ela é ótima para capturar estrutura espacial (uma imagem parada), mas não tem noção de **ordem temporal** -- não sabe distinguir "isto veio antes daquilo". Para sequências (texto, séries temporais, áudio), precisamos de outra arquitetura: a RNN.

## 3. Redes Neurais Recorrentes (RNN)

### O que é uma RNN?

As RNNs são redes neurais projetadas para processar **sequências de dados**, como séries temporais, texto ou áudio. Diferente da FFNN e da CNN, elas mantêm um **estado oculto** que é atualizado a cada passo de tempo, funcionando como uma memória de curto prazo do que já foi visto na sequência.

### Estrutura de uma RNN

1. **Camada de Entrada**: recebe um valor a cada passo de tempo.
2. **Camada Oculta Recorrente**: mantém um estado que é atualizado a cada passo, combinando a entrada atual com a memória do passo anterior.
3. **Camada de Saída**: produz a previsão final.

### Exemplo prático

Vamos prever o próximo valor de uma sequência numérica crescente (simulando, por exemplo, uma métrica de negócio subindo aos poucos). Diferente do exemplo original com uma única sequência `[1, 2, 3, 4, 5]`, aqui geramos 500 sequências diferentes, com pontos de partida e passos aleatórios, normalizadas entre 0 e 1 -- essencial para o treino convergir bem.

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Input
from sklearn.model_selection import train_test_split

seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)

n_amostras = 500
T = 5

inicio = np.random.uniform(0, 0.5, n_amostras)
passo = np.random.uniform(0.02, 0.1, n_amostras)

X_seq = np.array([[inicio[i] + passo[i] * t for t in range(T)] for i in range(n_amostras)])
y_seq = inicio + passo * T  # proximo valor da sequencia
X_seq = X_seq.reshape(n_amostras, T, 1)

X_treino, X_teste, y_treino, y_teste = train_test_split(X_seq, y_seq, test_size=0.2, random_state=seed)

modelo_rnn = Sequential()
modelo_rnn.add(Input(shape=(T, 1)))
modelo_rnn.add(SimpleRNN(8, activation='tanh'))
modelo_rnn.add(Dense(1))
modelo_rnn.compile(optimizer='adam', loss='mse')

print("Treinando a RNN...")
modelo_rnn.fit(X_treino, y_treino, epochs=100, verbose=0)
erro_teste = modelo_rnn.evaluate(X_teste, y_teste, verbose=0)
print(f"Erro medio quadratico (MSE) no teste: {erro_teste:.6f}")

previsoes = modelo_rnn.predict(X_teste[:5], verbose=0).flatten()
print("Previsoes:", previsoes.round(3))
print("Valores reais:", y_teste[:5].round(3))

Treinando a RNN...


Erro medio quadratico (MSE) no teste: 0.000068
Previsoes: [0.599 0.759 0.154 0.27  0.616]
Valores reais: [0.603 0.757 0.146 0.272 0.602]


### Explicação do Código

- Geramos 500 sequências diferentes (não apenas uma), cada uma com 5 passos de tempo, normalizadas entre 0 e 1.
- `SimpleRNN(8, activation='tanh')`: 8 unidades recorrentes, cada uma mantendo um estado que é atualizado a cada passo de tempo da sequência.
- O erro no conjunto de teste (sequências nunca vistas no treino) mostra se a rede aprendeu o padrão geral ("some um passo à frente"), e não apenas decorou os exemplos de treino.

**Aplicação real:** veja [`aplicacoes-de-negocio/03-previsao-demanda.ipynb`](aplicacoes-de-negocio/03-previsao-demanda.ipynb) para um exemplo de série temporal de negócio.

**Limitação da RNN simples:** o estado oculto vai perdendo informação das entradas mais antigas à medida que a sequência cresce -- um problema conhecido como **desvanecimento do gradiente**. Para sequências longas, onde o que aconteceu muitos passos atrás ainda importa, precisamos de uma versão com memória mais robusta: a LSTM.

## 4. Long Short-Term Memory (LSTM)

### O que é uma LSTM?

As LSTMs são um tipo especial de RNN (HOCHREITER; SCHMIDHUBER, 1997) projetado especificamente para lidar com o desvanecimento do gradiente. Elas usam "portões" (gates) que decidem, a cada passo, o que lembrar e o que esquecer, permitindo reter informação por sequências muito mais longas do que uma RNN simples.

### Estrutura de uma LSTM

1. **Portão de Esquecimento (Forget Gate)**: decide quais informações do estado anterior devem ser descartadas.
2. **Portão de Entrada (Input Gate)**: decide quais novas informações devem ser armazenadas.
3. **Portão de Saída (Output Gate)**: decide o que passar adiante para a saída.
4. **Célula de Memória**: armazena informações relevantes ao longo do tempo, protegida dos portões de entrada/esquecimento.

### Exemplo prático

Vamos repetir a mesma tarefa de previsão de sequência da seção anterior, mas com sequências mais longas (15 passos em vez de 5) -- um cenário mais favorável para a LSTM mostrar sua vantagem de memória mais longa.

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from sklearn.model_selection import train_test_split

seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)

n_amostras = 500
T = 15

inicio = np.random.uniform(0, 0.3, n_amostras)
passo = np.random.uniform(0.01, 0.05, n_amostras)

X_seq = np.array([[inicio[i] + passo[i] * t for t in range(T)] for i in range(n_amostras)])
y_seq = inicio + passo * T
X_seq = X_seq.reshape(n_amostras, T, 1)

X_treino, X_teste, y_treino, y_teste = train_test_split(X_seq, y_seq, test_size=0.2, random_state=seed)

modelo_lstm = Sequential()
modelo_lstm.add(Input(shape=(T, 1)))
modelo_lstm.add(LSTM(8))
modelo_lstm.add(Dense(1))
modelo_lstm.compile(optimizer='adam', loss='mse')

print("Treinando a LSTM...")
modelo_lstm.fit(X_treino, y_treino, epochs=100, verbose=0)
erro_teste = modelo_lstm.evaluate(X_teste, y_teste, verbose=0)
print(f"Erro medio quadratico (MSE) no teste: {erro_teste:.6f}")

previsoes = modelo_lstm.predict(X_teste[:5], verbose=0).flatten()
print("Previsoes:", previsoes.round(4))
print("Valores reais:", y_teste[:5].round(4))

Treinando a LSTM...


Erro medio quadratico (MSE) no teste: 0.000032
Previsoes: [0.6178 0.7701 0.1986 0.3033 0.5027]
Valores reais: [0.615  0.7679 0.1866 0.2992 0.4947]


### Explicação do Código

A estrutura do código é quase idêntica à da RNN -- trocamos apenas `SimpleRNN` por `LSTM` -- mas internamente a LSTM mantém três portões adicionais que regulam o fluxo de informação pela célula de memória.

**Uma nota honesta:** em testes diretos de RNN simples contra LSTM nesta mesma tarefa, com poucos dados e otimizadores modernos como o Adam, a RNN simples às vezes performa tão bem quanto a LSTM em sequências curtas -- a vantagem da LSTM fica mais evidente em sequências muito mais longas e problemas mais complexos do que os que cabem neste notebook didático. O ganho da LSTM é mais sobre **robustez em sequências longas e complexas** do que sobre acurácia garantida em qualquer tarefa.

**Aplicação real:** LSTMs são usadas em tradução automática, geração de texto e previsão de séries temporais mais longas do que a do exemplo acima.

**Limitação de RNN/LSTM:** ambas aprendem uma função que **prevê ou classifica** a partir de dados existentes. Elas não servem para **gerar dados novos e realistas** do zero. Para isso, existe outra família de arquiteturas: as GANs.

## 5. Redes Neurais Generativas Adversariais (GANs)

### O que são GANs?

As GANs (GOODFELLOW et al., 2014) consistem em **dois modelos** treinados simultaneamente, competindo entre si:

1. **Gerador**: recebe ruído aleatório e tenta gerar dados sintéticos que pareçam reais.
2. **Discriminador**: tenta distinguir os dados reais dos dados gerados.

O treinamento é um jogo de soma zero: o gerador melhora tentando enganar o discriminador, e o discriminador melhora tentando não se deixar enganar.

### Exemplo prático

Vamos gerar números sintéticos que se pareçam com uma distribuição uniforme entre 0 e 1. Diferente do exemplo original (que usava taxas de aprendizado iguais para as duas redes e não verificava o resultado), vamos aplicar desde o início as duas correções de estabilidade que já validamos no notebook de aplicações de negócio: **taxa de aprendizado bem menor no discriminador** e **label smoothing** (rótulos `0.9`/`0.1` em vez de `1.0`/`0.0`).

In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)

latent_dim = 5
input_dim = 1

gerador = Sequential([Input(shape=(latent_dim,)), Dense(16, activation='relu'), Dense(input_dim, activation='sigmoid')])
discriminador = Sequential([Input(shape=(input_dim,)), Dense(16, activation='relu'), Dense(1, activation='sigmoid')])

# Correcao 1: taxa de aprendizado bem menor no discriminador
discriminador.compile(optimizer=Adam(learning_rate=0.0002, beta_1=0.5), loss='binary_crossentropy', metrics=['accuracy'])
discriminador.trainable = False

gan = Sequential([gerador, discriminador])
gan.compile(optimizer=Adam(learning_rate=0.001, beta_1=0.5), loss='binary_crossentropy')

dados_reais = np.random.uniform(0, 1, (1000, input_dim))
epochs = 1000
batch_size = 32

for epoch in range(epochs):
    ruido = np.random.normal(0, 1, (batch_size, latent_dim))
    dados_falsos = gerador.predict(ruido, verbose=0)
    idx = np.random.randint(0, dados_reais.shape[0], batch_size)
    dados_reais_batch = dados_reais[idx]

    # Correcao 2: label smoothing (0.9 / 0.1 em vez de 1.0 / 0.0)
    y_real = np.ones((batch_size, 1)) * 0.9
    y_falso = np.ones((batch_size, 1)) * 0.1

    discriminador.trainable = True
    d_loss_real = discriminador.train_on_batch(dados_reais_batch, y_real)
    d_loss_falso = discriminador.train_on_batch(dados_falsos, y_falso)
    d_loss = 0.5 * np.add(d_loss_real, d_loss_falso)
    discriminador.trainable = False

    ruido = np.random.normal(0, 1, (batch_size, latent_dim))
    y_gan = np.ones((batch_size, 1))
    g_loss = gan.train_on_batch(ruido, y_gan)

    if (epoch + 1) % 200 == 0:
        print(f"Epoca {epoch+1}: perda discriminador={d_loss[0]:.4f} | perda gerador={g_loss:.4f}")

ruido_final = np.random.normal(0, 1, (200, latent_dim))
gerados = gerador.predict(ruido_final, verbose=0)
print()
print(f"Media dos dados reais: {dados_reais.mean():.4f} | desvio-padrao: {dados_reais.std():.4f}")
print(f"Media dos dados gerados: {gerados.mean():.4f} | desvio-padrao: {gerados.std():.4f}")

Epoca 200: perda discriminador=0.7060 | perda gerador=0.6262


Epoca 400: perda discriminador=0.7037 | perda gerador=0.6498


Epoca 600: perda discriminador=0.6985 | perda gerador=0.6775


Epoca 800: perda discriminador=0.6937 | perda gerador=0.7049


Epoca 1000: perda discriminador=0.6900 | perda gerador=0.7298

Media dos dados reais: 0.4903 | desvio-padrao: 0.2920
Media dos dados gerados: 0.9687 | desvio-padrao: 0.0226


**Veredito honesto:** as perdas do discriminador e do gerador ficam na mesma faixa (~0,68 a ~0,70) do início ao fim do treino, sem nenhuma das duas "vencer" a disputa de forma descontrolada -- exatamente o comportamento estável que buscamos. Isso não significa, porém, que o gerador aprendeu a distribuição perfeitamente: a média e o desvio-padrão dos números gerados ainda diferem dos dados reais. Um treino estável é uma condição necessária para uma boa geração, mas não suficiente sozinha -- mais épocas, mais dados ou uma rede maior ajudariam a fechar essa diferença.

**Aplicação real:** veja [`aplicacoes-de-negocio/10-geracao-conteudo-marketing.ipynb`](aplicacoes-de-negocio/10-geracao-conteudo-marketing.ipynb), de onde vieram as duas correções usadas aqui.

**Limitação das GANs:** elas geram dados novos, mas não produzem uma representação compacta e reversível de um dado existente. Para isso -- comprimir e reconstruir --, existe outra arquitetura: o Autoencoder.

## 6. Autoencoders

### O que são Autoencoders?

Autoencoders são redes neurais usadas para **aprender representações compactas** de dados, treinadas para reconstruir sua própria entrada. Têm duas partes:

1. **Encoder**: comprime a entrada em uma representação de menor dimensão (o "gargalo").
2. **Decoder**: reconstrói a entrada original a partir dessa representação comprimida.

São usados para redução de dimensionalidade, remoção de ruído (denoising) e detecção de anomalias (dados que o autoencoder reconstrói mal são candidatos a anomalias).

### Exemplo prático

O exemplo original usava números aleatórios sem nenhuma relação entre si -- um autoencoder não tem o que aprender aí, porque não existe estrutura para comprimir. Desta vez, vamos gerar 3 características **correlacionadas** (derivadas de um único fator latente comum, como aconteceria com dados reais de negócio) e comprimir para 1 dimensão, verificando se o autoencoder recupera essa estrutura.

In [5]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.model_selection import train_test_split

seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)

n = 1000
fator_latente = np.random.uniform(0, 1, n)
X_ae = np.column_stack([
    fator_latente + np.random.normal(0, 0.03, n),
    fator_latente * 0.8 + 0.1 + np.random.normal(0, 0.03, n),
    1 - fator_latente + np.random.normal(0, 0.03, n),
])
X_ae = np.clip(X_ae, 0, 1)

X_treino, X_teste = train_test_split(X_ae, test_size=0.2, random_state=seed)

input_dim = 3
encoding_dim = 1

entrada = Input(shape=(input_dim,))
codificacao = Dense(encoding_dim, activation='linear')(entrada)
decodificacao = Dense(input_dim, activation='sigmoid')(codificacao)
autoencoder = Model(entrada, decodificacao)
autoencoder.compile(optimizer='adam', loss='mse')

print("Treinando o autoencoder...")
autoencoder.fit(X_treino, X_treino, epochs=100, batch_size=32, verbose=0)

erro_teste = autoencoder.evaluate(X_teste, X_teste, verbose=0)
print(f"Erro de reconstrucao (MSE) no teste: {erro_teste:.6f}")

reconstruido = autoencoder.predict(X_teste[:5], verbose=0)
print("Original:")
print(X_teste[:5].round(3))
print("Reconstruido:")
print(reconstruido.round(3))

Treinando o autoencoder...


Erro de reconstrucao (MSE) no teste: 0.001180
Original:
[[0.411 0.375 0.591]
 [0.836 0.687 0.196]
 [0.499 0.475 0.552]
 [0.376 0.437 0.648]
 [0.91  0.874 0.071]]
Reconstruido:
[[0.363 0.393 0.636]
 [0.802 0.739 0.209]
 [0.461 0.467 0.541]
 [0.364 0.394 0.634]
 [0.9   0.837 0.11 ]]


### Explicação do Código

- As 3 características de entrada compartilham um único `fator_latente` -- exatamente o tipo de estrutura redundante que um autoencoder consegue comprimir.
- `Dense(encoding_dim, activation='linear')`: o encoder comprime as 3 características em apenas 1 número (o gargalo).
- `Dense(input_dim, activation='sigmoid')`: o decoder tenta reconstruir as 3 características originais a partir desse único número.
- O baixo erro de reconstrução no conjunto de teste mostra que o autoencoder aprendeu a capturar o fator latente comum, e não apenas "decorou" os dados de treino.

**Aplicação real:** veja [`aplicacoes-de-negocio/11-prevencao-lavagem-dinheiro.ipynb`](aplicacoes-de-negocio/11-prevencao-lavagem-dinheiro.ipynb), que usa exatamente essa ideia (transações que o autoencoder reconstrói mal são as candidatas a anomalia).

## Resumo dos Tipos de Redes Neurais

| **Tipo de Rede** | **Resolve a limitação de** | **Aplicação Principal** | **Exemplo neste notebook** |
|---|---|---|---|
| **FFNN** (Aula 3) | -- (arquitetura base) | Classificação, regressão | Detecção de spam |
| **CNN** | FFNN ignora vizinhança espacial | Imagens | Linha vertical vs. horizontal |
| **RNN** | CNN ignora ordem temporal | Sequências curtas | Previsão de próximo valor |
| **LSTM** | RNN esquece sequências longas | Sequências longas | Mesma tarefa, sequência mais longa |
| **GAN** | RNN/LSTM só preveem, não geram | Geração de dados | Números sintéticos |
| **Autoencoder** | GAN não comprime dados existentes | Redução de dimensionalidade, anomalias | Compressão de 3 features correlacionadas |

**Pergunta de síntese:** se quiséssemos aplicar alguma dessas arquiteturas especializadas ao problema original de detecção de spam (Aulas 1 a 3) -- por exemplo, analisando o **texto completo** do e-mail em vez de só 3 características já calculadas -- qual arquitetura você escolheria, e por quê?

# **Referências**

GOODFELLOW, I. et al. Generative adversarial networks. In: ADVANCES IN NEURAL INFORMATION PROCESSING SYSTEMS, 27., 2014. **Anais [...]**. p. 2672-2680, 2014.

HINTON, G. E.; SALAKHUTDINOV, R. R. Reducing the dimensionality of data with neural networks. **Science**, [s. l.], v. 313, n. 5786, p. 504-507, 2006.

HOCHREITER, S.; SCHMIDHUBER, J. Long short-term memory. **Neural Computation**, [s. l.], v. 9, n. 8, p. 1735-1780, 1997.